## 주문 상세 금액 계산

### 목적
주문 상세(order_items)의 수량과 단가를 곱해 항목별 결제 금액을 계산한다.

### 실행 전 예상
order_items에 item_amount라는 새 컬럼이 추가되고, 행 수는 원본과 동일할 것이다.

In [2]:
import pandas as pd
from pathlib import Path

# 노트북 위치 기준으로 프로젝트 루트 찾기
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

order_items = pd.read_csv(DATA_DIR / "order_items.csv")
order_items["item_amount"] = order_items["quantity"] * order_items["unit_price"]
order_items.head()

,order_item_id,order_id,product_id,quantity,unit_price,item_amount
0,1,1,216,4,13600,54400
1,2,1,34,1,87000,87000
2,3,1,187,1,93000,93000
3,4,1,112,1,225000,225000
4,5,2,214,1,64000,64000


### 실제 결과
item_amount 컬럼이 정상적으로 추가되었고, head()로 확인한 상위 5개 행에서
quantity * unit_price 값과 item_amount 값이 일치함을 확인했다.

### 검증 방법
order_items 전체 행 수가 원본과 동일한지, 
그리고 임의의 한 행에서 quantity * unit_price 계산값과 item_amount가 같은지 확인했다.

### AI 코드 수정 내용
**AI가 처음 제안한 코드:**

```python
order_items = pd.read_csv("data/raw/order_items.csv")
```

**문제:** 노트북 실행 위치가 notebooks/ 폴더 기준이라 상대경로가 파일을 찾지 못하는 FileNotFoundError 발생

**수정한 코드:**

```python
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
order_items = pd.read_csv(DATA_DIR / "order_items.csv")
```

**수정 이유:** 노트북이 어떤 위치에서 실행되든 항상 프로젝트 루트 기준으로 데이터 폴더를 찾도록 경로를 고정함

In [3]:
# 검증: item_amount가 실제로 quantity * unit_price와 같은지 확인
sample = order_items.iloc[0]
assert sample["item_amount"] == sample["quantity"] * sample["unit_price"]
print("검증 통과: item_amount 계산이 정확합니다.")

검증 통과: item_amount 계산이 정확합니다.


## 데이터 병합

### 목적
orders, order_items, products를 연결해 주문별 상품/카테고리 정보를 포함한 통합 데이터를 만든다.

### 실행 전 예상
병합 후 행 수는 order_items 원본 행 수와 같아야 하고, category 컬럼에 결측치가 없어야 한다.

In [9]:
# orders, products 데이터 불러오기
orders = pd.read_csv(DATA_DIR / "orders.csv")
products = pd.read_csv(DATA_DIR / "products.csv")

# 병합 전 원본 행 수 저장
before_rows = len(order_items)

# 1. order_items + orders 병합 (order_id 기준)
merged = order_items.merge(orders, on="order_id", how="left")

# 2. 위 결과 + products 병합 (product_id 기준)
merged = merged.merge(products, on="product_id", how="left")

# 병합 후 행 수 확인
after_rows = len(merged)

print(f"병합 전 행 수: {before_rows}")
print(f"병합 후 행 수: {after_rows}")
print(f"행 수 일치 여부: {before_rows == after_rows}")

# category 컬럼 결측치 확인
missing_category = merged["category"].isna().sum()
print(f"category 결측치 개수: {missing_category}")

merged.head()

병합 전 행 수: 14603
병합 후 행 수: 14603
행 수 일치 여부: True
category 결측치 개수: 0


,order_item_id,order_id,product_id,quantity,unit_price,item_amount,customer_id,order_date,payment_method,order_status,product_name,category,price
0,1,1,216,4,13600,54400,1121,2026-05-18,간편결제,배송중,컴팩트 에세이 그린 P216,도서,16000
1,2,1,34,1,87000,87000,1121,2026-05-18,간편결제,배송중,클래식 클렌징 폼 화이트 P034,뷰티,87000
2,3,1,187,1,93000,93000,1121,2026-05-18,간편결제,배송중,데일리 러닝 벨트 그레이 P187,스포츠,93000
3,4,1,112,1,225000,225000,1121,2026-05-18,간편결제,배송중,플러스 핸디 청소기 블루 P112,생활가전,250000
4,5,2,214,1,64000,64000,993,2026-02-01,신용카드,환불,스마트 클렌징 폼 화이트 P214,뷰티,64000
